In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import ruptures as rpt
import multiprocessing
from concurrent import futures
from tqdm import tqdm

import sys

sys.path.append("..")

from src import IOFunctions
from IPython.display import display, clear_output

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter(dark_background=False)

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

In [ ]:
dye_data_folder = '/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/Brendan/20260202_HollidayJunctions/200mMMgCl2'

In [ ]:
loc_files = H_F.file_search(dye_data_folder, ".h5", "")
print(f"Found {len(loc_files)} H5 files")
for i, f in enumerate(loc_files):
    print(f"  [{i}]: {os.path.basename(f)}")

In [ ]:
notch_filter = "semrock-nf03-405-488-561-635e"
dichroic_mirror = "semrock-di03-r488-561-t1-25x36"
dichroic_mirror_all = "semrock-di03-r405-488-561-635-t1-25x36"
shortpass_filter = "semrock-bsp01-785r"
longpass_red_filter = "semrock-blp01-635r"

longpass_filter = "semrock-blp01-568r"
filters = [dichroic_mirror]
dye = "Cy3"
average_emission_wavelengths, dye_pixel_efficiency = (
    S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
)
dye_pixel_efficiency_Cy3 = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

dye = "Cy5"
average_emission_wavelengths, dye_pixel_efficiency = (
    S_F.get_pixel_fractions_dye_and_filters(dye, filters, wavelength, pixel_QYs)
)
dye_pixel_efficiency_Cy5 = dye_pixel_efficiency / np.sum(dye_pixel_efficiency)

print(f"Cy3 pixel efficiencies (B, G, R): {dye_pixel_efficiency_Cy3}")
print(f"Cy5 pixel efficiencies (B, G, R): {dye_pixel_efficiency_Cy5}")

In [ ]:
def filter_initial_guess(df, tol=0.01):
    """Remove datapoints where A_R, A_G, and A_B are all within tol of 0.33.
    
    These represent the initial guess that was never updated by the fitter.
    
    Args:
        df (pd.DataFrame): DataFrame with A_R, A_G, A_B columns.
        tol (float): Tolerance for comparison to 0.33.
    
    Returns:
        pd.DataFrame: Filtered DataFrame with initial guesses removed.
    """
    initial_guess_mask = (
        np.isclose(df['A_R'], 0.33, atol=tol) &
        np.isclose(df['A_G'], 0.33, atol=tol) &
        np.isclose(df['A_B'], 0.33, atol=tol)
    )
    n_removed = initial_guess_mask.sum()
    n_total = len(df)
    print(f"Removed {n_removed}/{n_total} initial guess datapoints ({100*n_removed/n_total:.1f}%)")
    return df[~initial_guess_mask].copy()

In [ ]:
def find_colour_CPs(A_R, A_G, model="l2", min_size=5, jump=1):
    """Find change points jointly in A_R and A_G using multivariate detection.

    A_R and A_G are anti-correlated spectral fractions (A_R + A_G + A_B = 1),
    so a FRET transition shifts both simultaneously. Joint detection captures
    this correlated change rather than treating channels independently.

    Uses BIC penalty (log(n) * sigma^2) rather than the n * sigma^2 used for
    photoelectron traces — spectral fractions are bounded [0, 1] so the
    original penalty is orders of magnitude too conservative.

    Args:
        A_R (np.ndarray): Red spectral fraction time series.
        A_G (np.ndarray): Green spectral fraction time series.
        model (str): Cost model for ruptures.
        min_size (int): Minimum segment size.
        jump (int): Grid of change point candidates.

    Returns:
        list: Change point indices (last element is always len(signal)).
    """
    n = len(A_R)
    if n < min_size:
        return [n]
    # stack into 2D array (n_samples, 2) for multivariate detection
    signal = np.column_stack([A_R, A_G])
    dim = signal.shape[1]
    # estimate noise from per-channel variance
    sigma2 = np.nanmean([np.nanvar(A_R), np.nanvar(A_G)])
    if sigma2 == 0:
        return [n]
    # BIC penalty: log(n) * dim * sigma^2
    pen = np.log(n) * dim * sigma2
    algo = rpt.Pelt(model=model, min_size=min_size, jump=jump).fit(signal)
    my_CPs = algo.predict(pen=pen)
    return my_CPs


def find_CPs_for_puncta(args):
    """Find change points for a single punctum jointly on A_R and A_G.

    Args:
        args (tuple): (puncta_id, A_R_values, A_G_values)

    Returns:
        tuple: (puncta_id, CPs, has_changepoint)
    """
    puncta_id, A_R, A_G = args
    CPs = find_colour_CPs(A_R, A_G)
    # has changepoint if more than 1 CP
    # (ruptures always reports the final index as a CP)
    has_cp = len(CPs) > 1
    return (puncta_id, CPs, has_cp)


def find_CPs_parallel(df):
    """Run joint change point detection on all puncta in parallel.

    Args:
        df (pd.DataFrame): Filtered DataFrame with puncta_id, A_R, A_G.

    Returns:
        dict: {puncta_id: CPs} for puncta with at least one change point.
    """
    puncta_ids = df['puncta_id'].unique()

    # prepare tasks
    tasks = []
    for pid in puncta_ids:
        subset = df[df['puncta_id'] == pid].sort_values('frame')
        A_R = subset['A_R'].values.astype(np.float64)
        A_G = subset['A_G'].values.astype(np.float64)
        tasks.append((pid, A_R, A_G))

    n_workers = min(60, max(1, int(0.9 * multiprocessing.cpu_count())))
    cp_results = {}

    with futures.ProcessPoolExecutor(n_workers) as executor:
        fs = {executor.submit(find_CPs_for_puncta, task): task[0] for task in tasks}
        with tqdm(desc="Finding colour CPs", total=len(tasks), unit="punctum") as pbar:
            for f in futures.as_completed(fs):
                pbar.update()
                puncta_id, CPs, has_cp = f.result()
                if has_cp:
                    cp_results[puncta_id] = CPs

    print(f"\nPuncta with change points: {len(cp_results)}/{len(puncta_ids)} "
          f"({100*len(cp_results)/len(puncta_ids):.1f}%)")
    return cp_results

## Load data and apply filters

In [ ]:
# load data - change index as needed
file_idx = 1
print(f"Loading: {os.path.basename(loc_files[file_idx])}")
df = pd.read_hdf(loc_files[file_idx])
print(f"Total datapoints: {len(df)}")
print(f"Unique puncta: {df['puncta_id'].nunique()}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# step 1: remove datapoints where A_R, A_G, A_B are all ~0.33 (initial guess)
df_filtered = filter_initial_guess(df, tol=0.01)
print(f"Remaining datapoints: {len(df_filtered)}")
print(f"Remaining puncta: {df_filtered['puncta_id'].nunique()}")

In [ ]:
# step 2: find change points in A_R and A_G channels
cp_results = find_CPs_parallel(df_filtered)

In [ ]:
# step 3: keep only puncta with at least one change point
cp_puncta_ids = list(cp_results.keys())
df_cp = df_filtered[df_filtered['puncta_id'].isin(cp_puncta_ids)].copy()
print(f"Puncta with change points: {len(cp_puncta_ids)}")
print(f"Datapoints in filtered dataset: {len(df_cp)}")

## Visualise puncta with change points

In [ ]:
exposure_time = 0.1  # seconds per frame

for pid in sorted(cp_puncta_ids):
    subset = df_cp[df_cp['puncta_id'] == pid].sort_values('frame')
    if len(subset) < 5:
        continue

    CPs = cp_results[pid]
    t = subset['frame'].values * exposure_time

    fig, axs = plotter.two_column_plot(ncolumns=3, widthratio=[1, 1, 1])
    fig.suptitle(f"Puncta {pid}  ({len(CPs)-1} change point(s))", fontsize=10)

    # panel 1: A_R and A_G vs time with change points
    axs[0].scatter(t, subset['A_R'].values, s=5, color='red', label='A_R')
    axs[0].scatter(t, subset['A_G'].values, s=5, color='green', label='A_G')
    axs[0].scatter(t, subset['A_B'].values, s=5, color='blue', label='A_B', alpha=0.3)
    # mark joint change points
    frames_sorted = subset['frame'].values
    for cp_idx in CPs[:-1]:  # exclude final index
        if cp_idx < len(frames_sorted):
            axs[0].axvline(x=frames_sorted[cp_idx] * exposure_time, color='black', ls='--', alpha=0.7, lw=1.5)
    axs[0].set_xlabel('time / s')
    axs[0].set_ylabel('spectral fraction')
    axs[0].set_ylim([0, 1])
    axs[0].legend(fontsize=6, markerscale=2)

    # panel 2: R/G ratio vs time
    ratio = subset['A_R'].values / subset['A_G'].values
    axs[1] = plotter.scatter_plot(
        axs[1], x=t, y=ratio, xaxislabel='time / s',
        yaxislabel='R/G ratio', facecolor='red'
    )
    # mark change points on ratio plot too
    for cp_idx in CPs[:-1]:
        if cp_idx < len(frames_sorted):
            axs[1].axvline(x=frames_sorted[cp_idx] * exposure_time, color='black', ls='--', alpha=0.7, lw=1.5)
    xmin, xmax = axs[1].get_xlim()
    axs[1].hlines(
        y=dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1],
        xmin=xmin, xmax=xmax, ls='--', color='darkred', label='Cy5'
    )
    axs[1].hlines(
        y=dye_pixel_efficiency_Cy3[2] / dye_pixel_efficiency_Cy3[1],
        xmin=xmin, xmax=xmax, ls='--', color='orange', label='Cy3'
    )
    axs[1].set_xlim([xmin, xmax])
    ymin, ymax = axs[1].get_ylim()
    if dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1] > ymax:
        axs[1].set_ylim(ymin, 1.1 * (dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1]))
    axs[1].legend(fontsize=6)

    # panel 3: R/G ratio vs photons
    photons = subset['A_R'].values + subset['A_G'].values + subset['A_B'].values
    axs[2] = plotter.scatter_plot(
        axs[2], x=photons, y=ratio, xaxislabel='photons',
        yaxislabel='R/G ratio', facecolor='green'
    )
    xmin, xmax = axs[2].get_xlim()
    axs[2].hlines(
        y=dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1],
        xmin=xmin, xmax=xmax, ls='--', color='darkred'
    )
    axs[2].hlines(
        y=dye_pixel_efficiency_Cy3[2] / dye_pixel_efficiency_Cy3[1],
        xmin=xmin, xmax=xmax, ls='--', color='orange'
    )
    axs[2].set_xlim([xmin, xmax])
    ymin, ymax = axs[2].get_ylim()
    if dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1] > ymax:
        axs[2].set_ylim(ymin, 1.1 * (dye_pixel_efficiency_Cy5[2] / dye_pixel_efficiency_Cy5[1]))

    clear_output(wait=True)
    plt.show()
    plt.pause(5)

In [ ]:
# summary statistics
print("=" * 60)
print("Post-hoc analysis summary")
print("=" * 60)
print(f"Total datapoints loaded:              {len(df)}")
print(f"Removed (initial guess ~0.33):         {len(df) - len(df_filtered)}")
print(f"Total puncta:                          {df['puncta_id'].nunique()}")
print(f"Puncta with change points (A_R+A_G):   {len(cp_puncta_ids)}")
print(f"Datapoints in final dataset:           {len(df_cp)}")
print()
# breakdown of CP counts
n_cps = [len(cp_results[pid]) - 1 for pid in cp_puncta_ids]
print(f"Change points per punctum (joint): mean={np.mean(n_cps):.1f}, max={np.max(n_cps)}")